In [1]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [2]:
# load directory 

In [3]:
DATA_PATH = 'data/'
def load_pdf_files(data):
    loader = DirectoryLoader(data,
                             glob='*.pdf',
                             loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

In [4]:
documents = load_pdf_files(data=DATA_PATH)
print('length of pdf pages:',len(documents))

length of pdf pages: 759


In [5]:
def create_chunks(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500,
                                                   chunk_overlap = 50)
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks

In [6]:
text_chunks = create_chunks(extracted_data=documents)
print("length of text_chunks:",len(text_chunks))

length of text_chunks: 7079


In [7]:
def get_embedding_model():
    embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L12-v2")
    return embedding_model

In [8]:
embedding_model = get_embedding_model()
embedding_model

d:\projects\general_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L12-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [9]:
# store embedds in FAISS

In [10]:
DB_FAISS_PATH = "vectorstore/db_faiss"
db = FAISS.from_documents(text_chunks,embedding_model)
db.save_local(DB_FAISS_PATH)

# 

################# connect memmory

In [11]:
# set llm (mistral with hugging face)

In [12]:
import os
from langchain_huggingface import HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain_huggingface import HuggingFaceEmbeddings

from dotenv import load_dotenv
load_dotenv()

True

In [13]:
HF_TOKEN = os.getenv("HF_TOKEN")

In [15]:
HUGGINGFACE_REPO_ID = "mistralai/Mistral-7B-Instruct-v0.3"
#HUGGINGFACE_REPO_ID="mistralai/Mistral-7B-Instruct-v0.2"
def load_llm(HUGGINGFACE_REPO_ID):
    llm = HuggingFaceEndpoint(
        repo_id =HUGGINGFACE_REPO_ID,
        temperature = 0.5,
        model_kwargs = {"token":HF_TOKEN,
        'max_length':"512"}
    )
    return llm




In [16]:
# connect llm with FAISS

In [17]:
custom_prompt_template = """
use the pieces of information provides in the context to answer user's question.
if you don't know the answer, just say that you don't know,don't try to make up an answer. Don't provide anything out of the given context.

context :{context}
Question : {question}
start the answer directly. No small talk please.
"""

In [18]:


def set_custom_prompt(custom_prompt_template):
    prompt = PromptTemplate(template = custom_prompt_template,input_variables ={"context","Question"} )
    return prompt

In [19]:
DB_FAISS_PATH = "vectorstore/db_faiss"
embedding_model = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L12-v2")
db = FAISS.load_local(DB_FAISS_PATH,embedding_model,allow_dangerous_deserialization = True)


In [20]:
# create chain

In [21]:
qa_chain = RetrievalQA.from_chain_type(
    llm = load_llm(HUGGINGFACE_REPO_ID),
    chain_type = "stuff",
    retriever = db.as_retriever(search_kwargs = {'k':3}),
    return_source_documents = True,
    chain_type_kwargs = {'prompt':set_custom_prompt(custom_prompt_template)}
)

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to C:\Users\anjuc\.cache\huggingface\token
Login successful


In [22]:
#invoke with single query

user_query = input("Write a query here...")
response = qa_chain.invoke({'query':user_query})
print(response)


{'query': 'reason for cancer', 'result': "\nThe major reasons for cancer are tobacco and alcohol use, diet, sexual and reproductive behavior, infectious agents, family history, occupation, environment and pollution. Many cancers are caused by changes in the cell's DNA due to damage from the environment, which are known as carcinogens. Specific carcinogens have been difficult to identify, but dietary factors seem to be involved. For example, colon cancer is more common in industrialized nations and diets high in fat, red meat, total calories, and alcohol seem to predispose.", 'source_documents': [Document(id='8f88f903-2340-4f96-b5b1-3a0441e4f984', metadata={'source': 'data\\The_GALE_ENCYCLOPEDIA_of_MEDICINE_SECOND.pdf', 'page': 20}, page_content='Causes and symptoms\nThe major risk factors for cancer are: tobacco, alco-\nhol, diet, sexual and reproductive behavior, infectious\nagents, family history, occupation, environment and pol-\nlution.\nAccording to the estimates of the American C

In [23]:
response['result']

"\nThe major reasons for cancer are tobacco and alcohol use, diet, sexual and reproductive behavior, infectious agents, family history, occupation, environment and pollution. Many cancers are caused by changes in the cell's DNA due to damage from the environment, which are known as carcinogens. Specific carcinogens have been difficult to identify, but dietary factors seem to be involved. For example, colon cancer is more common in industrialized nations and diets high in fat, red meat, total calories, and alcohol seem to predispose."

In [29]:
pages = [doc.metadata['page'] for doc in response['source_documents']]
pages

[20, 20, 237]